# 04 — Analisi quantitativa degli errori e robustezza per D3

Questo notebook risponde alla domanda di ricerca **D3** analizzando quantitativamente la robustezza dei modelli transfer rispetto a sottotipo tumorale, fattore di ingrandimento e paziente. Utilizza le predizioni per immagine già generate dal notebook 03 e non esegue training, non carica modelli e non modifica subset, split o fold.

L'analisi confronta le prestazioni nei sottogruppi, studia falsi positivi e falsi negativi, valuta la confidenza degli errori e verifica se gli errori si concentrano in pazienti specifici. Non produce Grad-CAM: gli output quantitativi vengono salvati in `results/04_d3_error_analysis/` e costituiscono un riferimento anche per il successivo approfondimento qualitativo di explainability e robustezza.

## 1. Setup ambiente, Google Drive e path

In Colab il notebook monta Google Drive e usa `COLAB_PROJECT_ROOT` come cartella del progetto; questo valore va modificato soltanto se il progetto si trova in una posizione diversa su Drive. In locale la root viene individuata automaticamente risalendo dalla directory corrente.

`PROJECT_ROOT` è la base di tutti i path successivi. Tabelle, figure ed esempi di errore prodotti da questa analisi vengono salvati in modo persistente sotto `results/04_d3_error_analysis/`.

In [1]:
from pathlib import Path
import os
import sys

# La disponibilità del modulo google.colab distingue Colab dall'esecuzione locale.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Modificare soltanto questo valore se il progetto si trova in un'altra cartella di Drive.
COLAB_PROJECT_ROOT = "/content/drive/MyDrive/HistoBreastNet"

# In Colab Drive viene montato per rendere persistenti input e output tra le sessioni.
if IN_COLAB:
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(COLAB_PROJECT_ROOT).resolve()

    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(
            f"PROJECT_ROOT non esiste: {PROJECT_ROOT}\n"
            "Aggiorna COLAB_PROJECT_ROOT con il percorso corretto su Google Drive."
        )

else:
    # In locale risale dalla directory corrente finché trova la struttura attesa del progetto.
    current = Path.cwd().resolve()
    candidates = [current] + list(current.parents)

    PROJECT_ROOT = None
    for candidate in candidates:
        if (candidate / "notebooks").exists() and (candidate / "data").exists():
            PROJECT_ROOT = candidate
            break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Root del progetto non trovata. Esegui il notebook dalla cartella del progetto "
            "oppure imposta PROJECT_ROOT manualmente."
        )

# Tutti i path successivi sono costruiti a partire da questa root comune.
os.chdir(PROJECT_ROOT)

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Directory di lavoro corrente:", os.getcwd())
print("Esecuzione in Colab:", IN_COLAB)

PROJECT_ROOT: /Users/bertu/Projects/HistoBreastNet
Directory di lavoro corrente: /Users/bertu/Projects/HistoBreastNet
Esecuzione in Colab: False


### Import, directory di input e directory di output

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Root dei dati del progetto.
DATA_DIR = PROJECT_ROOT / "data"
# Directory del subset patient-level già preparato nelle fasi precedenti.
PROCESSED_DIR = DATA_DIR / "processed" / "diversity_1p5GB"
# Root condivisa degli output prodotti dai diversi notebook.
RESULTS_DIR = PROJECT_ROOT / "results"
# Predizioni transfer per immagine generate dal notebook 03.
PREDICTIONS_PATH = RESULTS_DIR / "03_transfer_learning" / "predictions" / "transfer_learning_predictions.csv"
# Metadati di supporto, letti soltanto se non sono già inclusi nelle predizioni.
METADATA_PATH = PROCESSED_DIR / "metadata_subset.csv"
# Tutti gli output del notebook 04 restano confinati in questa directory.
OUT_DIR = RESULTS_DIR / "04_d3_error_analysis"
# CSV quantitativi: metriche aggregate, sottogruppi ed elenchi di errori.
TABLES_DIR = OUT_DIR / "tables"
# Figure in PNG/PDF prodotte dalle analisi quantitative.
FIGURES_DIR = OUT_DIR / "figures"
# Copie tabellari degli esempi FP/FN destinate alla successiva ispezione qualitativa.
EXAMPLES_DIR = OUT_DIR / "examples"

for directory in (OUT_DIR, TABLES_DIR, FIGURES_DIR, EXAMPLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Directory di output:", OUT_DIR)

PROJECT_ROOT: /Users/bertu/Projects/HistoBreastNet
Directory di output: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis


## 2. Caricamento delle predizioni e dei metadati

L'analisi usa le predizioni out-of-fold del protocollo k-fold patient-wise. Se il CSV contiene anche altri protocolli, vengono mantenute soltanto le righe coerenti con la valutazione finale.

Gli eventuali metadati mancanti vengono recuperati da `metadata_subset.csv` mediante un merge many-to-one su `relative_path`: ogni immagine può comparire in più righe di predizione, ma deve corrispondere a una sola riga di metadati. Le colonne già presenti nelle predizioni vengono conservate senza duplicarle.

In [3]:
# Controllo bloccante: senza predizioni per immagine non è possibile svolgere D3.
if not PREDICTIONS_PATH.is_file():
    raise FileNotFoundError(f"Predizioni transfer non trovate: {PREDICTIONS_PATH}")

# Il CSV contiene probabilità e metadati delle run transfer già completate nel notebook 03.
predictions = pd.read_csv(PREDICTIONS_PATH)
# Se coesistono più protocolli, conserva soltanto le predizioni out-of-fold finali.
if "split_type" in predictions.columns:
    before = len(predictions)
    predictions = predictions.loc[
        predictions["split_type"].astype(str).str.strip().str.lower().eq("kfold_patient_wise")
    ].copy()
    print(f"Predizioni filtrate per kfold_patient_wise: {len(predictions)} / {before}")
# Un risultato vuoto indicherebbe un input incompatibile, quindi l'analisi viene interrotta.
if predictions.empty:
    raise ValueError(
        "Non rimangono predizioni transfer dopo la selezione del protocollo kfold_patient_wise."
    )

# Metadati necessari per analisi per sottotipo, ingrandimento, paziente e classe.
important_metadata = [
    "subtype", "subtype_name", "magnification", "patient_id", "label", "binary_label"
]
# Vengono recuperate soltanto le colonne assenti, evitando duplicati con suffissi _x/_y.
missing_metadata = [column for column in important_metadata if column not in predictions.columns]

# Il merge è necessario solo quando il CSV del notebook 03 non contiene già tutti i metadati.
if missing_metadata:
    if "relative_path" not in predictions.columns:
        raise KeyError(
            "Le predizioni non contengono alcune colonne di metadati e non possono essere "
            "arricchite perché 'relative_path' è assente."
        )
    if not METADATA_PATH.is_file():
        raise FileNotFoundError(f"Metadati richiesti ma non trovati: {METADATA_PATH}")
    metadata = pd.read_csv(METADATA_PATH)
    if "relative_path" not in metadata.columns:
        raise KeyError("Merge dei metadati non riuscito: 'relative_path' è assente da metadata_subset.csv.")
    unavailable = [column for column in missing_metadata if column not in metadata.columns]
    if unavailable:
        raise KeyError(f"Colonne di metadati richieste non disponibili: {unavailable}")
    # relative_path deve identificare una sola immagine nei metadati; duplicati renderebbero
    # ambiguo l'arricchimento e potrebbero moltiplicare artificialmente le predizioni.
    if metadata["relative_path"].duplicated().any():
        raise ValueError("metadata_subset.csv contiene valori relative_path duplicati; il merge è ambiguo.")
    merge_columns = ["relative_path", *missing_metadata]
    # validate="many_to_one" consente più configurazioni predittive per la stessa immagine,
    # ma impone una sola riga metadata per relative_path.
    predictions = predictions.merge(
        metadata[merge_columns], on="relative_path", how="left", validate="many_to_one"
    )
    # Nessuna riga può restare priva dei metadati richiesti dopo il merge.
    if predictions[missing_metadata].isna().any(axis=None):
        missing_rows = int(predictions[missing_metadata].isna().any(axis=1).sum())
        raise ValueError(f"Il merge dei metadati ha lasciato incomplete {missing_rows} righe di predizione.")
    print(f"Colonne di metadati unite: {missing_metadata}")
else:
    print("Il CSV delle predizioni contiene già tutti i metadati richiesti.")

# y_prob è indispensabile per AUROC, confidenza e ricostruzione della classe a soglia 0.5.
if "y_prob" not in predictions.columns:
    raise KeyError("Le predizioni devono contenere 'y_prob' per l'analisi quantitativa degli errori.")
predictions["y_prob"] = pd.to_numeric(predictions["y_prob"], errors="raise").astype(float)
# Il controllo [0, 1] evita analisi probabilistiche su score non validi o valori mancanti.
if predictions["y_prob"].isna().any() or not predictions["y_prob"].between(0.0, 1.0).all():
    raise ValueError("Tutti i valori di y_prob devono essere probabilità finite comprese tra 0 e 1.")

# Preferisce il target esplicito y_true; binary_label è il fallback equivalente.
if "y_true" in predictions.columns:
    predictions["y_true"] = pd.to_numeric(predictions["y_true"], errors="raise").astype(int)
elif "binary_label" in predictions.columns:
    predictions["y_true"] = pd.to_numeric(predictions["binary_label"], errors="raise").astype(int)
else:
    raise KeyError("Le predizioni devono contenere 'y_true' oppure 'binary_label'.")

# Usa la decisione già salvata quando presente; altrimenti applica la soglia standard 0.5.
if "y_pred" in predictions.columns:
    predictions["y_pred"] = pd.to_numeric(predictions["y_pred"], errors="raise").astype(int)
else:
    predictions["y_pred"] = (predictions["y_prob"] >= 0.5).astype(int)

for column in ("binary_label", "fold"):
    if column in predictions.columns:
        predictions[column] = pd.to_numeric(predictions[column], errors="raise").astype(int)

# Questi controlli bloccanti garantiscono una codifica binaria coerente prima delle metriche.
if not predictions["y_true"].isin([0, 1]).all():
    raise ValueError("y_true deve contenere solo valori binari 0 e 1.")
if not predictions["y_pred"].isin([0, 1]).all():
    raise ValueError("y_pred deve contenere solo valori binari 0 e 1.")

def normalize_magnification(value):
    """Uniforma l'ingrandimento nel formato canonico 40X, 100X, 200X o 400X."""
    if pd.isna(value):
        return pd.NA
    token = str(value).strip().upper().replace("×", "X")
    number = token[:-1] if token.endswith("X") else token
    try:
        return f"{int(float(number))}X"
    except ValueError as exc:
        raise ValueError(f"Valore di magnification non valido: {value!r}") from exc

if "magnification" not in predictions.columns:
    raise KeyError("L'analisi richiede la colonna 'magnification'.")
# La normalizzazione evita che forme equivalenti, ad esempio 40, 40x e 40×, creino gruppi distinti.
predictions["magnification"] = predictions["magnification"].map(normalize_magnification).astype("string")

required_analysis_columns = ["model", "training_mode", "subtype_name", "patient_id"]
missing_required = [column for column in required_analysis_columns if column not in predictions.columns]
# Le colonne seguenti identificano configurazione e sottogruppi; senza di esse D3 non è definibile.
if missing_required:
    raise KeyError(f"Nelle predizioni mancano colonne richieste per l'analisi: {missing_required}")

print("Dimensione predizioni:", predictions.shape)
display(predictions.head())

Predizioni filtrate per kfold_patient_wise: 11352 / 11352
Il CSV delle predizioni contiene già tutti i metadati richiesti.
Dimensione predizioni: (11352, 17)


,experiment_id,dataset_config,split_type,fold,model,training_mode,relative_path,filename,patient_id,binary_label,label,subtype,subtype_name,magnification,y_true,y_prob,y_pred
0,20260705_135337_375681_diversity_1p5GB_kfold_p...,diversity_1p5GB,kfold_patient_wise,0,mobilenetv2,frozen,histology_slides/breast/benign/SOB/adenosis/SO...,SOB_B_A-14-22549AB-100-009.png,SOB_B_A_14-22549AB,0,benign,A,adenosis,100X,0,0.343230,0
1,20260705_135337_375681_diversity_1p5GB_kfold_p...,diversity_1p5GB,kfold_patient_wise,0,mobilenetv2,frozen,histology_slides/breast/benign/SOB/adenosis/SO...,SOB_B_A-14-22549AB-100-029.png,SOB_B_A_14-22549AB,0,benign,A,adenosis,100X,0,0.187969,0
2,20260705_135337_375681_diversity_1p5GB_kfold_p...,diversity_1p5GB,kfold_patient_wise,0,mobilenetv2,frozen,histology_slides/breast/benign/SOB/adenosis/SO...,SOB_B_A-14-22549AB-100-012.png,SOB_B_A_14-22549AB,0,benign,A,adenosis,100X,0,0.055281,0
3,20260705_135337_375681_diversity_1p5GB_kfold_p...,diversity_1p5GB,kfold_patient_wise,0,mobilenetv2,frozen,histology_slides/breast/benign/SOB/adenosis/SO...,SOB_B_A-14-22549AB-100-015.png,SOB_B_A_14-22549AB,0,benign,A,adenosis,100X,0,0.193766,0
4,20260705_135337_375681_diversity_1p5GB_kfold_p...,diversity_1p5GB,kfold_patient_wise,0,mobilenetv2,frozen,histology_slides/breast/benign/SOB/adenosis/SO...,SOB_B_A-14-22549AB-100-003.png,SOB_B_A_14-22549AB,0,benign,A,adenosis,100X,0,0.557174,1


## 3. Selezione dei modelli da analizzare

Il confronto generale include le quattro configurazioni transfer: MobileNetV2 ed EfficientNetB0, ciascuno in modalità frozen e fine-tuned. **EfficientNetB0 fine-tuned** è il modello principale selezionato per le analisi dettagliate di robustezza, errori, confidenza e distribuzione patient-level.

In [4]:
# Quattro configurazioni previste: due backbone × stage frozen/fine-tuned.
TRANSFER_SPECS = [
    ("mobilenetv2", "frozen"),
    ("mobilenetv2", "finetuned"),
    ("efficientnetb0", "frozen"),
    ("efficientnetb0", "finetuned"),
]
# Modello transfer finale usato nelle analisi dettagliate di errori e robustezza.
BEST_MODEL = ("efficientnetb0", "finetuned")

# La normalizzazione testuale rende confrontabili etichette con maiuscole o spazi diversi.
predictions["model"] = predictions["model"].astype(str).str.strip().str.lower()
predictions["training_mode"] = predictions["training_mode"].astype(str).str.strip().str.lower()
# Verifica quali configurazioni sono effettivamente presenti nel CSV aggregato.
available_specs = set(zip(predictions["model"], predictions["training_mode"]))
missing_specs = [spec for spec in TRANSFER_SPECS if spec not in available_specs]
# Una configurazione assente produce un avviso ma non blocca le analisi ancora possibili.
if missing_specs:
    print(f"Attenzione: nessuna riga di predizione trovata per le configurazioni transfer: {missing_specs}")

# Esclude eventuali righe estranee alle quattro configurazioni transfer principali.
transfer_mask = pd.Series(
    list(zip(predictions["model"], predictions["training_mode"])), index=predictions.index
).isin(TRANSFER_SPECS)
transfer_predictions = predictions.loc[transfer_mask].copy()
if transfer_predictions.empty:
    raise ValueError("Nessuna riga corrisponde alle configurazioni transfer richieste.")
# Ogni immagine deve contribuire una sola volta per configurazione alle metriche out-of-fold.
if "relative_path" in transfer_predictions.columns:
    duplicated_transfer = transfer_predictions.duplicated(
        subset=["model", "training_mode", "relative_path"]
    ).sum()
    if duplicated_transfer:
        raise ValueError(
            f"Le predizioni transfer contengono {duplicated_transfer} righe model/training_mode/relative_path duplicate. "
            "È attesa una sola predizione out-of-fold per immagine e configurazione transfer."
        )

# Sottoinsieme del modello selezionato per tutte le analisi granulari successive.
best_predictions = transfer_predictions.loc[
    (transfer_predictions["model"] == BEST_MODEL[0])
    & (transfer_predictions["training_mode"] == BEST_MODEL[1])
].copy()
if best_predictions.empty:
    raise ValueError(f"Nessuna predizione trovata per BEST_MODEL={BEST_MODEL}.")
# Il controllo impedisce che una stessa immagine pesi due volte in errori, heatmap o pazienti.
if "relative_path" in best_predictions.columns:
    duplicated = best_predictions.duplicated(subset=["relative_path"]).sum()
    if duplicated:
        raise ValueError(
            f"Le predizioni del modello selezionato contengono {duplicated} righe relative_path duplicate. "
            "È attesa una sola predizione out-of-fold per immagine."
        )

print("Righe transfer:", len(transfer_predictions))
print("Righe predizioni del modello selezionato:", len(best_predictions))

Righe transfer: 11352
Righe predizioni del modello selezionato: 2838


## 4. Funzioni di supporto per metriche, tabelle e figure

In [5]:
# Ordine fisico degli ingrandimenti, riutilizzato in tabelle e heatmap.
MAGNIFICATION_ORDER = ["40X", "100X", "200X", "400X"]
METRIC_COLUMNS = [
    "n", "accuracy", "precision", "recall", "f1", "auroc", "specificity",
    "recall_malignant", "recall_benign", "tn", "fp", "fn", "tp",
]


def compute_binary_metrics(df):
    """Calcola metriche binarie per immagine usando malignant come classe positiva."""
    # Un gruppo vuoto mantiene lo schema: conteggi a zero e metriche non calcolabili a NaN.
    if df.empty:
        return {column: (0 if column in {"n", "tn", "fp", "fn", "tp"} else np.nan)
                for column in METRIC_COLUMNS}
    y_true = df["y_true"].astype(int).to_numpy()
    y_pred = df["y_pred"].astype(int).to_numpy()
    y_prob = df["y_prob"].astype(float).to_numpy()
    # labels=[0, 1] forza sempre l'ordine benigno/maligno: la matrice resta 2×2
    # anche quando nel sottogruppo manca una classe, evitando errori durante ravel().
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    # Sensibilità maligna: quota di casi maligni correttamente riconosciuti.
    recall_malignant = tp / (tp + fn) if (tp + fn) else np.nan
    # Recall benigna: quota di benigni corretti, equivalente alla specificità.
    recall_benign = tn / (tn + fp) if (tn + fp) else np.nan
    # L'AUROC richiede sia benigni sia maligni per confrontare il ranking delle probabilità.
    # Nei gruppi mono-classe vale NaN: significa «non calcolabile», non prestazione zero.
    auroc = roc_auc_score(y_true, y_prob) if np.unique(y_true).size == 2 else np.nan
    return {
        "n": int(len(df)),
        "accuracy": accuracy_score(y_true, y_pred),
        # zero_division=0 rende robuste precision, recall e F1 nei gruppi piccoli in cui
        # non esistono predizioni positive o il denominatore della metrica è nullo.
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auroc": auroc,
        "specificity": recall_benign,
        "recall_malignant": recall_malignant,
        "recall_benign": recall_benign,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def grouped_metrics(df, group_columns):
    """Applica le metriche binarie separatamente a ciascun sottogruppo richiesto."""
    rows = []
    # Pandas accetta una colonna singola o una lista: la chiave viene adattata senza
    # cambiare il significato del raggruppamento.
    group_key = group_columns[0] if len(group_columns) == 1 else group_columns
    # Le metriche vengono calcolate indipendentemente per ogni sottotipo, ingrandimento,
    # paziente o combinazione di colonne richiesta dal chiamante.
    for keys, group in df.groupby(group_key, dropna=False, sort=False):
        keys = (keys,) if len(group_columns) == 1 else keys
        row = dict(zip(group_columns, keys))
        row.update(compute_binary_metrics(group))
        rows.append(row)
    return pd.DataFrame(rows, columns=[*group_columns, *METRIC_COLUMNS])


def save_table(df, filename):
    """Salva una tabella CSV nella directory tables/ e restituisce il path."""
    # Tutte le tabelle confluiscono nella stessa directory senza alterarne nome o colonne.
    path = TABLES_DIR / filename
    df.to_csv(path, index=False)
    print(f"Salvato: {path}")
    return path


def save_figure(fig, stem):
    """Salva la stessa figura in PNG e PDF nella directory figures/."""
    fig.tight_layout()
    for extension in ("png", "pdf"):
        path = FIGURES_DIR / f"{stem}.{extension}"
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print(f"Salvato: {path}")
    plt.close(fig)


def horizontal_bar_plot(df, category, value, title, xlabel, stem, color="#35618f"):
    """Ordina e visualizza una metrica per sottogruppo mediante barre orizzontali."""
    # L'ordinamento crescente porta in evidenza i gruppi peggiori, utile per individuare
    # rapidamente sottotipi o ingrandimenti critici; i NaN restano distinti dai valori zero.
    plot_df = df.sort_values(value, ascending=True, na_position="first")
    height = max(4.0, 0.45 * len(plot_df) + 1.5)
    fig, ax = plt.subplots(figsize=(9, height))
    bars = ax.barh(plot_df[category].astype(str), plot_df[value], color=color)
    ax.set(title=title, xlabel=xlabel, ylabel="")
    if value not in {"n", "error_rate"}:
        ax.set_xlim(0, 1.05)
    ax.grid(axis="x", alpha=0.25)
    for bar, number in zip(bars, plot_df[value]):
        if pd.notna(number):
            label = f"{int(number)}" if value == "n" else f"{number:.3f}"
            ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f" {label}", va="center")
    save_figure(fig, stem)

## 5. Confronto complessivo tra modelli transfer

Le metriche vengono calcolate aggregando le predizioni out-of-fold di ciascuna coppia modello/modalità di training. Questa sezione offre una vista complessiva delle quattro configurazioni; le sezioni successive approfondiscono sottotipi, ingrandimenti, pazienti ed errori.

In [6]:
# Aggrega tutte le predizioni out-of-fold di ciascuna configurazione transfer.
overall_transfer_metrics = grouped_metrics(transfer_predictions, ["model", "training_mode"])
# Impone lo stesso ordine di TRANSFER_SPECS per rendere tabella e grafico confrontabili.
spec_order = {spec: index for index, spec in enumerate(TRANSFER_SPECS)}
overall_transfer_metrics["_order"] = [
    spec_order.get((model, mode), len(spec_order))
    for model, mode in zip(overall_transfer_metrics["model"], overall_transfer_metrics["training_mode"])
]
overall_transfer_metrics = overall_transfer_metrics.sort_values("_order").drop(columns="_order")
save_table(overall_transfer_metrics, "overall_transfer_metrics.csv")

# Metriche principali: prestazione generale, equilibrio, ranking e sensibilità maligna.
metrics_to_plot = ["accuracy", "f1", "auroc", "recall_malignant"]
labels = overall_transfer_metrics["model"] + "\n" + overall_transfer_metrics["training_mode"]
x = np.arange(len(overall_transfer_metrics))
width = 0.19
fig, ax = plt.subplots(figsize=(11, 6))
for index, metric in enumerate(metrics_to_plot):
    ax.bar(x + (index - 1.5) * width, overall_transfer_metrics[metric], width, label=metric)
ax.set(title="Prestazioni complessive out-of-fold dei modelli transfer", ylabel="Punteggio", xticks=x, xticklabels=labels)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.25)
ax.legend(ncol=2)
save_figure(fig, "overall_transfer_metrics")
display(overall_transfer_metrics)

Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/overall_transfer_metrics.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/overall_transfer_metrics.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/overall_transfer_metrics.pdf


,model,training_mode,n,accuracy,precision,recall,f1,auroc,specificity,recall_malignant,recall_benign,tn,fp,fn,tp
0,mobilenetv2,frozen,2838,0.618746,0.609575,0.642502,0.625606,0.670121,0.595388,0.642502,0.595388,852,579,503,904
1,mobilenetv2,finetuned,2838,0.613108,0.605605,0.629709,0.617422,0.668406,0.596785,0.629709,0.596785,854,577,521,886
2,efficientnetb0,frozen,2838,0.724806,0.704575,0.766169,0.734082,0.788361,0.684137,0.766169,0.684137,979,452,329,1078
3,efficientnetb0,finetuned,2838,0.738548,0.714655,0.786780,0.748985,0.803710,0.691125,0.786780,0.691125,989,442,300,1107


## 6. Analisi per sottotipo tumorale

I sottotipi di BreakHis sono spesso mono-classe: un sottotipo può contenere soltanto immagini benigne oppure soltanto immagini maligne. Di conseguenza, l'AUROC non è definita nei gruppi con una sola classe e viene riportata come `NaN`.

Per i sottotipi benigni la metrica più informativa è `recall_benign`, equivalente alla specificità; per quelli maligni è `recall_malignant`, cioè la sensibilità sulla classe clinicamente critica. `relevant_class_recall` seleziona automaticamente la recall coerente con la classe del sottotipo. La F1 della classe positiva viene comunque salvata come vista secondaria, ma è poco informativa nei sottotipi esclusivamente benigni. Tutte le metriche devono essere lette insieme al supporto `n`, perché i gruppi piccoli producono stime meno stabili.

In [7]:
# Prima tabella: confronta tutte le configurazioni transfer sugli stessi sottotipi.
subtype_metrics_all_models = grouped_metrics(
    transfer_predictions, ["model", "training_mode", "subtype_name"]
)
save_table(subtype_metrics_all_models, "subtype_metrics_all_models.csv")

# Seconda tabella: si concentra sul modello finale selezionato per l'analisi dettagliata.
subtype_metrics_best_model = grouped_metrics(best_predictions, ["subtype_name"])
# In BreakHis molti sottotipi sono mono-classe: il groupby controlla tutte le y_true
# e assegna al sottotipo l'appartenenza benigna, maligna o mista.
subtype_class = best_predictions.groupby("subtype_name", dropna=False)["y_true"].agg(
    lambda values: "malignant" if values.eq(1).all() else ("benign" if values.eq(0).all() else "mixed")
).rename("class_membership").reset_index()
# class_membership viene affiancata alle metriche per decidere quale recall sia interpretabile.
subtype_metrics_best_model = subtype_metrics_best_model.merge(subtype_class, on="subtype_name", how="left")
# Per sottotipi benigni usa recall_benign; per quelli maligni usa recall_malignant.
# Questa metrica evita di leggere F1/recall positiva come se fossero informative anche
# quando il sottotipo non contiene alcuna immagine maligna.
subtype_metrics_best_model["relevant_class_recall"] = np.select(
    [
        subtype_metrics_best_model["class_membership"].eq("benign"),
        subtype_metrics_best_model["class_membership"].eq("malignant"),
    ],
    [subtype_metrics_best_model["recall_benign"], subtype_metrics_best_model["recall_malignant"]],
    # Nei gruppi misti non esiste una singola classe rilevante, quindi resta NaN.
    default=np.nan,
)
save_table(subtype_metrics_best_model, "subtype_metrics_best_model.csv")

# Vista principale e più confrontabile nei sottotipi mono-classe.
horizontal_bar_plot(
    subtype_metrics_best_model, "subtype_name", "relevant_class_recall",
    "Recall della classe rilevante per sottotipo — EfficientNetB0 fine-tuned",
    "Recall della classe del sottotipo",
    "subtype_relevant_class_recall_best_model", color="#287271",
)
# F1 positiva conservata come metrica secondaria; nei sottotipi benigni è poco informativa.
horizontal_bar_plot(
    subtype_metrics_best_model, "subtype_name", "f1",
    "F1 per sottotipo — EfficientNetB0 fine-tuned", "F1 della classe positiva",
    "subtype_f1_best_model",
)
# Sensibilità maligna mostrata esplicitamente per evidenziare i maligni non riconosciuti.
horizontal_bar_plot(
    subtype_metrics_best_model, "subtype_name", "recall_malignant",
    "Recall maligna per sottotipo — EfficientNetB0 fine-tuned", "Recall (maligna)",
    "subtype_recall_malignant_best_model", color="#a63d40",
)
# Il supporto n contestualizza la stabilità: pochi campioni rendono più variabile la recall.
horizontal_bar_plot(
    subtype_metrics_best_model, "subtype_name", "n",
    "Supporto per sottotipo — EfficientNetB0 fine-tuned", "Numero di immagini",
    "subtype_support_best_model", color="#587b4b",
)
display(subtype_metrics_best_model)

Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/subtype_metrics_all_models.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/subtype_metrics_best_model.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/subtype_relevant_class_recall_best_model.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/subtype_relevant_class_recall_best_model.pdf
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/subtype_f1_best_model.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/subtype_f1_best_model.pdf
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/subtype_recall_malignant_best_model.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/subtype_recall_malignant_best_model.pdf
Salvato: /Users/bertu/Projects/HistoBreastNet/results/

,subtype_name,n,accuracy,precision,recall,f1,auroc,specificity,recall_malignant,recall_benign,tn,fp,fn,tp,class_membership,relevant_class_recall
0,adenosis,444,0.682432,0.0,0.000000,0.000000,NaN,0.682432,NaN,0.682432,303,141,0,0,benign,0.682432
1,tubular_adenoma,247,0.914980,0.0,0.000000,0.000000,NaN,0.914980,NaN,0.914980,226,21,0,0,benign,0.914980
2,papillary_carcinoma,291,0.573883,1.0,0.573883,0.729258,NaN,NaN,0.573883,NaN,0,0,124,167,malignant,0.573883
3,ductal_carcinoma,219,0.945205,1.0,0.945205,0.971831,NaN,NaN,0.945205,NaN,0,0,12,207,malignant,0.945205
4,mucinous_carcinoma,271,0.575646,1.0,0.575646,0.730679,NaN,NaN,0.575646,NaN,0,0,115,156,malignant,0.575646
5,phyllodes_tumor,453,0.591611,0.0,0.000000,0.000000,NaN,0.591611,NaN,0.591611,268,185,0,0,benign,0.591611
6,fibroadenoma,287,0.668990,0.0,0.000000,0.000000,NaN,0.668990,NaN,0.668990,192,95,0,0,benign,0.668990
7,lobular_carcinoma,626,0.921725,1.0,0.921725,0.959268,NaN,NaN,0.921725,NaN,0,0,49,577,malignant,0.921725


## 7. Analisi per fattore di ingrandimento

L'analisi verifica se le prestazioni rimangono stabili passando tra 40X, 100X, 200X e 400X. L'ordine degli ingrandimenti viene imposto esplicitamente per evitare ordinamenti lessicografici non desiderati; anche in questo caso il supporto deve accompagnare l'interpretazione delle metriche.

In [8]:
# Vista globale: confronta tutte le configurazioni transfer sui quattro ingrandimenti.
magnification_metrics_all_models = grouped_metrics(
    transfer_predictions, ["model", "training_mode", "magnification"]
)
# Categorical usa MAGNIFICATION_ORDER per imporre 40X, 100X, 200X, 400X;
# senza categorie ordinate Pandas potrebbe adottare un ordine lessicografico non metodologico.
magnification_metrics_all_models["magnification"] = pd.Categorical(
    magnification_metrics_all_models["magnification"], categories=MAGNIFICATION_ORDER, ordered=True
)
magnification_metrics_all_models = magnification_metrics_all_models.sort_values(
    ["model", "training_mode", "magnification"]
)
save_table(magnification_metrics_all_models, "magnification_metrics_all_models.csv")

# Vista del modello finale: verifica se la robustezza cambia al variare della scala ottica.
magnification_metrics_best_model = grouped_metrics(best_predictions, ["magnification"])
# Applica lo stesso ordine anche alla tabella del modello selezionato, così grafici e CSV
# sono direttamente confrontabili con la vista di tutti i modelli.
magnification_metrics_best_model["magnification"] = pd.Categorical(
    magnification_metrics_best_model["magnification"], categories=MAGNIFICATION_ORDER, ordered=True
)
magnification_metrics_best_model = magnification_metrics_best_model.sort_values("magnification")
save_table(magnification_metrics_best_model, "magnification_metrics_best_model.csv")

# Il ciclo visualizza F1, AUROC, recall maligna e supporto per ingrandimento.
# Il supporto è indispensabile: metriche su pochi esempi sono meno stabili.
for metric, title, stem_metric, color in [
    ("f1", "F1", "f1", "#35618f"),
    ("auroc", "AUROC", "auroc", "#6f4e7c"),
    ("recall_malignant", "Recall maligna", "recall_malignant", "#a63d40"),
    ("n", "Supporto", "support", "#587b4b"),
]:
    horizontal_bar_plot(
        magnification_metrics_best_model, "magnification", metric,
        f"{title} per ingrandimento — EfficientNetB0 fine-tuned",
        "Numero di immagini" if metric == "n" else title,
        f"magnification_{stem_metric}_best_model", color=color,
    )
display(magnification_metrics_best_model)

Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/magnification_metrics_all_models.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/magnification_metrics_best_model.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/magnification_f1_best_model.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/magnification_f1_best_model.pdf
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/magnification_auroc_best_model.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/magnification_auroc_best_model.pdf
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/magnification_recall_malignant_best_model.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/magnification_recall_malignant_best_model.pdf
Salvato: /Users/bertu/Projects/HistoBr

,magnification,n,accuracy,precision,recall,f1,auroc,specificity,recall_malignant,recall_benign,tn,fp,fn,tp
1,40X,712,0.714888,0.702997,0.732955,0.717663,0.776491,0.697222,0.732955,0.697222,251,109,94,258
0,100X,743,0.749664,0.723150,0.812332,0.765152,0.817716,0.686486,0.812332,0.686486,254,116,70,303
2,200X,728,0.752747,0.738155,0.797844,0.766839,0.824685,0.705882,0.797844,0.705882,252,105,75,296
3,400X,655,0.735878,0.690608,0.803859,0.742942,0.793399,0.674419,0.803859,0.674419,232,112,61,250


## 8. Robustezza per sottotipo e ingrandimento

Ogni cella della tabella e delle heatmap rappresenta una specifica coppia sottotipo/ingrandimento. Sono mantenuti anche i gruppi con pochi campioni, che richiedono maggiore cautela interpretativa. L'AUROC è `NaN` quando una combinazione contiene una sola classe e un valore mancante non deve essere interpretato come zero.

La vista principale usa `relevant_class_recall`: recall benigna per i gruppi benigni e recall maligna per quelli maligni. La heatmap consente di individuare combinazioni specifiche nelle quali il modello risulta meno robusto; la tabella conserva anche `n` e le altre metriche per fornire il contesto necessario.

In [9]:
# Ogni gruppo corrisponde a una specifica coppia sottotipo × ingrandimento.
# La granularità maggiore può produrre celle con supporto ridotto e metriche più instabili.
subtype_magnification_metrics = grouped_metrics(
    best_predictions, ["subtype_name", "magnification"]
)
# Identifica la classe presente in ciascuna cella per scegliere la recall pertinente.
subtype_magnification_class = best_predictions.groupby(
    ["subtype_name", "magnification"], dropna=False
)["y_true"].agg(
    lambda values: "malignant" if values.eq(1).all() else ("benign" if values.eq(0).all() else "mixed")
).rename("class_membership").reset_index()
# one_to_one garantisce una sola riga di metriche e una sola classe per combinazione.
subtype_magnification_metrics = subtype_magnification_metrics.merge(
    subtype_magnification_class, on=["subtype_name", "magnification"], how="left", validate="one_to_one"
)
# relevant_class_recall evita di interpretare la F1 positiva come metrica primaria
# nei sottotipi benigni, dove la classe positiva non è presente.
subtype_magnification_metrics["relevant_class_recall"] = np.select(
    [
        subtype_magnification_metrics["class_membership"].eq("benign"),
        subtype_magnification_metrics["class_membership"].eq("malignant"),
    ],
    [
        subtype_magnification_metrics["recall_benign"],
        subtype_magnification_metrics["recall_malignant"],
    ],
    default=np.nan,
)
# Mantiene le colonne della futura heatmap nell'ordine ottico 40X→400X.
subtype_magnification_metrics["magnification"] = pd.Categorical(
    subtype_magnification_metrics["magnification"], categories=MAGNIFICATION_ORDER, ordered=True
)
subtype_magnification_metrics = subtype_magnification_metrics.sort_values(
    ["subtype_name", "magnification"]
)
save_table(subtype_magnification_metrics, "subtype_magnification_metrics_best_model.csv")

def plot_subtype_magnification_heatmap(metric, title, colorbar_label, stem):
    """Visualizza una metrica nella matrice sottotipo × ingrandimento."""
    # pivot trasforma la tabella lunga in matrice: righe = sottotipi,
    # colonne = ingrandimenti, celle = valore della metrica scelta.
    heatmap_data = subtype_magnification_metrics.pivot(
        index="subtype_name", columns="magnification", values=metric
    ).reindex(columns=MAGNIFICATION_ORDER).sort_index()
    fig_height = max(5.0, 0.65 * len(heatmap_data.index) + 2)
    fig, ax = plt.subplots(figsize=(9, fig_height))
    # masked_invalid nasconde i NaN: indicano una metrica non definita, non un errore
    # di elaborazione e soprattutto non una prestazione pari a zero.
    masked_values = np.ma.masked_invalid(heatmap_data.to_numpy(dtype=float))
    # vmin=0 e vmax=1 fissano la stessa scala per tutte le celle e per entrambe le heatmap,
    # rendendo i colori confrontabili senza riscalatura automatica.
    image = ax.imshow(masked_values, aspect="auto", vmin=0, vmax=1, cmap="viridis")
    ax.set(
        title=title, xlabel="Ingrandimento", ylabel="Sottotipo",
        xticks=np.arange(len(heatmap_data.columns)),
        yticks=np.arange(len(heatmap_data.index)),
        xticklabels=heatmap_data.columns, yticklabels=heatmap_data.index,
    )
    for row in range(masked_values.shape[0]):
        for column in range(masked_values.shape[1]):
            value = heatmap_data.iloc[row, column]
            # «N/D» significa non disponibile e compare quando la metrica non è calcolabile;
            # altrimenti il valore viene mostrato con due decimali.
            annotation = "N/D" if pd.isna(value) else f"{value:.2f}"
            # Il colore del testo cambia in funzione dell'intensità della cella:
            # bianco sul fondo scuro, nero sul fondo chiaro, per mantenere leggibilità.
            text_color = "white" if pd.notna(value) and value < 0.55 else "black"
            ax.text(column, row, annotation, ha="center", va="center", color=text_color, fontsize=9)
    fig.colorbar(image, ax=ax, label=colorbar_label)
    save_figure(fig, stem)


# Vista consigliata per individuare combinazioni specifiche difficili.
plot_subtype_magnification_heatmap(
    "relevant_class_recall",
    "Recall della classe rilevante per sottotipo × ingrandimento — EfficientNetB0 fine-tuned",
    "Recall della classe rilevante",
    "subtype_magnification_relevant_recall_heatmap_best_model",
)
# F1 positiva mantenuta come vista secondaria e da leggere insieme a classe e supporto.
plot_subtype_magnification_heatmap(
    "f1",
    "F1 per sottotipo × ingrandimento — EfficientNetB0 fine-tuned",
    "F1 della classe positiva",
    "subtype_magnification_f1_heatmap_best_model",
)
display(subtype_magnification_metrics)

Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/subtype_magnification_metrics_best_model.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/subtype_magnification_relevant_recall_heatmap_best_model.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/subtype_magnification_relevant_recall_heatmap_best_model.pdf
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/subtype_magnification_f1_heatmap_best_model.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/subtype_magnification_f1_heatmap_best_model.pdf


,subtype_name,magnification,n,accuracy,precision,recall,f1,auroc,specificity,recall_malignant,recall_benign,tn,fp,fn,tp,class_membership,relevant_class_recall
1,adenosis,40X,114,0.710526,0.0,0.000000,0.000000,NaN,0.710526,NaN,0.710526,81,33,0,0,benign,0.710526
0,adenosis,100X,113,0.681416,0.0,0.000000,0.000000,NaN,0.681416,NaN,0.681416,77,36,0,0,benign,0.681416
2,adenosis,200X,111,0.648649,0.0,0.000000,0.000000,NaN,0.648649,NaN,0.648649,72,39,0,0,benign,0.648649
3,adenosis,400X,106,0.688679,0.0,0.000000,0.000000,NaN,0.688679,NaN,0.688679,73,33,0,0,benign,0.688679
13,ductal_carcinoma,40X,45,0.933333,1.0,0.933333,0.965517,NaN,NaN,0.933333,NaN,0,0,3,42,malignant,0.933333
12,ductal_carcinoma,100X,54,0.981481,1.0,0.981481,0.990654,NaN,NaN,0.981481,NaN,0,0,1,53,malignant,0.981481
14,ductal_carcinoma,200X,66,0.909091,1.0,0.909091,0.952381,NaN,NaN,0.909091,NaN,0,0,6,60,malignant,0.909091
15,ductal_carcinoma,400X,54,0.962963,1.0,0.962963,0.981132,NaN,NaN,0.962963,NaN,0,0,2,52,malignant,0.962963
25,fibroadenoma,40X,78,0.679487,0.0,0.000000,0.000000,NaN,0.679487,NaN,0.679487,53,25,0,0,benign,0.679487
24,fibroadenoma,100X,74,0.621622,0.0,0.000000,0.000000,NaN,0.621622,NaN,0.621622,46,28,0,0,benign,0.621622


## 9. Analisi degli errori: falsi positivi e falsi negativi

Un falso positivo è un'immagine benigna classificata come maligna; un falso negativo è un'immagine maligna classificata come benigna. I falsi negativi sono particolarmente rilevanti perché rappresentano casi maligni non riconosciuti.

Le tabelle degli esempi ordinano gli errori più confidenti, cioè quelli per cui il modello assegna con maggiore sicurezza la classe sbagliata. Questi file supportano la successiva ispezione qualitativa senza introdurre nuove analisi in questo notebook.

In [10]:
# TP: maligno riconosciuto; TN: benigno riconosciuto; FP: benigno indicato come maligno;
# FN: maligno indicato come benigno, errore particolarmente critico clinicamente.
conditions = [
    # y_true=1 e y_pred=1: vero positivo (TP), maligno riconosciuto correttamente.
    best_predictions["y_true"].eq(1) & best_predictions["y_pred"].eq(1),
    # y_true=0 e y_pred=0: vero negativo (TN), benigno riconosciuto correttamente.
    best_predictions["y_true"].eq(0) & best_predictions["y_pred"].eq(0),
    # y_true=0 e y_pred=1: falso positivo (FP), benigno classificato come maligno.
    best_predictions["y_true"].eq(0) & best_predictions["y_pred"].eq(1),
    # y_true=1 e y_pred=0: falso negativo (FN), maligno non riconosciuto.
    best_predictions["y_true"].eq(1) & best_predictions["y_pred"].eq(0),
]
# np.select assegna a ogni immagine una sola categoria di esito.
best_predictions["error_type"] = np.select(conditions, ["TP", "TN", "FP", "FN"], default="UNKNOWN")
if best_predictions["error_type"].eq("UNKNOWN").any():
    raise ValueError("Combinazione y_true/y_pred inattesa durante l'assegnazione di error_type.")
# Confidenza = probabilità della classe effettivamente predetta.
# Confidenza elevata su FP/FN significa che il modello sbaglia in modo convinto.
best_predictions["confidence"] = np.maximum(best_predictions["y_prob"], 1.0 - best_predictions["y_prob"])
# Margini piccoli identificano immagini vicine alla soglia e quindi decisioni incerte.
# Margine = distanza dalla soglia 0.5; valori grandi indicano decisioni più nette.
best_predictions["margin"] = np.abs(best_predictions["y_prob"] - 0.5)
save_table(best_predictions, "best_model_predictions_with_error_type.csv")

# Ordine fisso per mantenere coerenti tabelle, boxplot e conteggi.
ERROR_TYPE_ORDER = ["TP", "TN", "FP", "FN"]


def error_counts_by(df, group_columns):
    """Conta TP, TN, FP e FN e calcola il tasso di errore per sottogruppo."""
    counts = (
        df.groupby([*group_columns, "error_type"], dropna=False)
        .size().unstack(fill_value=0).reindex(columns=ERROR_TYPE_ORDER, fill_value=0).reset_index()
    )
    counts["n"] = counts[ERROR_TYPE_ORDER].sum(axis=1)
    counts["errors"] = counts["FP"] + counts["FN"]
    counts["error_rate"] = counts["errors"] / counts["n"]
    return counts


error_counts_by_subtype = error_counts_by(best_predictions, ["subtype_name"])
error_counts_by_magnification = error_counts_by(best_predictions, ["magnification"])
error_counts_by_subtype_magnification = error_counts_by(
    best_predictions, ["subtype_name", "magnification"]
)
save_table(error_counts_by_subtype, "error_counts_by_subtype.csv")
save_table(error_counts_by_magnification, "error_counts_by_magnification.csv")
save_table(error_counts_by_subtype_magnification, "error_counts_by_subtype_magnification.csv")

# FP ordinati per y_prob decrescente: probabilità alte corrispondono a benigni
# classificati maligni con grande sicurezza, quindi prioritari per l'ispezione qualitativa.
false_positive_examples = best_predictions.loc[best_predictions["error_type"].eq("FP")].sort_values(
    "y_prob", ascending=False
).head(20)
# FN ordinati per y_prob crescente: probabilità basse corrispondono a maligni
# classificati benigni con grande sicurezza, errori clinicamente rilevanti da ispezionare.
false_negative_examples = best_predictions.loc[best_predictions["error_type"].eq("FN")].sort_values(
    "y_prob", ascending=True
).head(20)
save_table(false_positive_examples, "false_positive_examples.csv")
save_table(false_negative_examples, "false_negative_examples.csv")
# Le copie in examples/ agevolano l'ispezione qualitativa successiva degli errori più netti.
false_positive_examples.to_csv(EXAMPLES_DIR / "false_positive_examples.csv", index=False)
false_negative_examples.to_csv(EXAMPLES_DIR / "false_negative_examples.csv", index=False)
print(f"Salvato: {EXAMPLES_DIR / 'false_positive_examples.csv'}")
print(f"Salvato: {EXAMPLES_DIR / 'false_negative_examples.csv'}")

print("Conteggi degli errori:")
display(best_predictions["error_type"].value_counts().reindex(ERROR_TYPE_ORDER, fill_value=0))

Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/best_model_predictions_with_error_type.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/error_counts_by_subtype.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/error_counts_by_magnification.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/error_counts_by_subtype_magnification.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/false_positive_examples.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/false_negative_examples.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/examples/false_positive_examples.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/examples/false_negative_examples.csv
Conteggi degli errori:


error_type
TP    1107
TN     989
FP     442
FN     300
Name: count, dtype: int64

## 10. Analisi della confidenza

La confidenza è la probabilità assegnata alla classe predetta, mentre il margine misura la distanza dalla soglia decisionale 0.5. Errori ad alta confidenza indicano previsioni sbagliate ma nette; errori a bassa confidenza corrispondono invece a immagini vicine alla soglia.

In [11]:
# y_prob è la probabilità maligna: se il modello sceglie maligno la confidenza è y_prob,
# se sceglie benigno è 1-y_prob. Il massimo misura quindi la sicurezza nella classe scelta.
best_predictions["confidence"] = np.maximum(best_predictions["y_prob"], 1.0 - best_predictions["y_prob"])
# Il margine misura la distanza dalla soglia 0.5: valori piccoli indicano casi borderline,
# valori grandi decisioni nette, corrette oppure sbagliate.
best_predictions["margin"] = np.abs(best_predictions["y_prob"] - 0.5)
# is_correct separa predizioni corrette ed errate nei successivi istogrammi.
best_predictions["is_correct"] = best_predictions["y_pred"].eq(best_predictions["y_true"])

# Statistiche descrittive della confidenza per ciascun tipo di esito TP/TN/FP/FN.
confidence_summary = (
    best_predictions.groupby("error_type", dropna=False)
    .agg(
        n=("confidence", "size"),
        confidence_mean=("confidence", "mean"),
        confidence_std=("confidence", "std"),
        confidence_median=("confidence", "median"),
        confidence_min=("confidence", "min"),
        confidence_max=("confidence", "max"),
        margin_mean=("margin", "mean"),
        margin_median=("margin", "median"),
    ).reindex(ERROR_TYPE_ORDER).reset_index()
)
save_table(confidence_summary, "confidence_summary_by_error_type.csv")

# La confidenza è sempre in [0.5, 1]: sotto 0.5 sarebbe stata scelta l'altra classe.
# I 21 estremi producono intervalli regolari per confrontare corrette ed errate.
bins = np.linspace(0.5, 1.0, 21)
# Istogramma corrette vs errate: confronta quanto sono nette le decisioni corrette e gli errori.
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(best_predictions.loc[best_predictions["is_correct"], "confidence"], bins=bins, alpha=0.65, label="Corrette")
ax.hist(best_predictions.loc[~best_predictions["is_correct"], "confidence"], bins=bins, alpha=0.65, label="Errate")
ax.set(title="Confidenza: predizioni corrette vs errate", xlabel="Confidenza", ylabel="Numero di immagini")
ax.legend()
ax.grid(axis="y", alpha=0.25)
save_figure(fig, "confidence_correct_vs_wrong")

fig, ax = plt.subplots(figsize=(9, 5.5))
# Il boxplot mostra quali categorie di errore tendono a essere più confidenti.
box_data = [best_predictions.loc[best_predictions["error_type"].eq(kind), "confidence"] for kind in ERROR_TYPE_ORDER]
ax.boxplot(box_data, tick_labels=ERROR_TYPE_ORDER, showmeans=True)
ax.set(title="Confidenza per tipo di esito", xlabel="Esito", ylabel="Confidenza", ylim=(0.48, 1.02))
ax.grid(axis="y", alpha=0.25)
save_figure(fig, "confidence_by_error_type")

# Qui si osserva direttamente y_prob, non la confidenza: varia in [0, 1] e permette
# di confrontare la distribuzione dei benigni reali con quella dei maligni reali.
probability_bins = np.linspace(0, 1, 26)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(best_predictions.loc[best_predictions["y_true"].eq(0), "y_prob"], bins=probability_bins, alpha=0.65, label="Benigno reale")
ax.hist(best_predictions.loc[best_predictions["y_true"].eq(1), "y_prob"], bins=probability_bins, alpha=0.65, label="Maligno reale")
# 0.5 è la soglia decisionale: a sinistra il modello predice benigno,
# a destra predice maligno.
ax.axvline(0.5, color="black", linestyle="--", linewidth=1.2, label="Soglia decisionale")
ax.set(title="Probabilità predetta di malignità per classe reale", xlabel="y_prob", ylabel="Numero di immagini")
ax.legend()
ax.grid(axis="y", alpha=0.25)
save_figure(fig, "probability_distribution_by_true_class")
display(confidence_summary)

Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/confidence_summary_by_error_type.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/confidence_correct_vs_wrong.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/confidence_correct_vs_wrong.pdf
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/confidence_by_error_type.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/confidence_by_error_type.pdf
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/probability_distribution_by_true_class.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/probability_distribution_by_true_class.pdf


,error_type,n,confidence_mean,confidence_std,confidence_median,confidence_min,confidence_max,margin_mean,margin_median
0,TP,1107,0.839487,0.135794,0.882258,0.500976,0.998738,0.339487,0.382258
1,TN,989,0.820011,0.145369,0.855969,0.500401,0.999475,0.320011,0.355969
2,FP,442,0.748698,0.150956,0.745492,0.500409,0.998036,0.248698,0.245492
3,FN,300,0.737942,0.146477,0.729046,0.504810,0.997408,0.237942,0.229046


## 11. Analisi patient-level

La valutazione è patient-wise: le immagini appartenenti a ciascun paziente non visto vengono considerate congiuntamente per verificare se gli errori si concentrano in pochi soggetti. Il tasso di errore dei pazienti con poche immagini è meno stabile e deve essere interpretato con cautela.

In [12]:
# Calcola le metriche separatamente per ogni paziente non visto: consente di verificare
# se una quota rilevante degli errori è concentrata in pochi soggetti difficili.
patient_level_metrics = grouped_metrics(best_predictions, ["patient_id"])
# errors conta tutte le classificazioni sbagliate del paziente: FP + FN.
patient_level_metrics["errors"] = patient_level_metrics["fp"] + patient_level_metrics["fn"]
# error_rate normalizza gli errori per n, rendendo confrontabili pazienti con supporti diversi.
patient_level_metrics["error_rate"] = patient_level_metrics["errors"] / patient_level_metrics["n"]
# Ordinamento decrescente: porta in alto i pazienti con error_rate maggiore;
# a parità di tasso privilegia quelli con più immagini e quindi una stima più stabile.
patient_level_metrics = patient_level_metrics.sort_values(
    ["error_rate", "n"], ascending=[False, False]
)
save_table(patient_level_metrics, "patient_level_metrics_best_model.csv")

# Per il grafico l'ordine viene invertito, così le barre orizzontali crescono dal basso verso l'alto.
plot_patients = patient_level_metrics.sort_values("error_rate", ascending=True)
fig_height = max(7.0, 0.26 * len(plot_patients) + 2)
fig, ax = plt.subplots(figsize=(10, fig_height))
# I pazienti con meno di 10 immagini sono distinti in arancione: su pochi esempi anche
# una singola previsione errata può cambiare molto il tasso osservato.
colors = np.where(plot_patients["n"].lt(10), "#c9843e", "#35618f")
ax.barh(plot_patients["patient_id"].astype(str), plot_patients["error_rate"], color=colors)
ax.set(
    title="Tasso di errore per paziente — EfficientNetB0 fine-tuned",
    xlabel="Tasso di errore (arancione: meno di 10 immagini)", ylabel="Paziente", xlim=(0, 1.05),
)
ax.grid(axis="x", alpha=0.25)
save_figure(fig, "patient_error_rate_best_model")
display(patient_level_metrics.head(20))

Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/patient_level_metrics_best_model.csv
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/patient_error_rate_best_model.png
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/figures/patient_error_rate_best_model.pdf


,patient_id,n,accuracy,precision,recall,f1,auroc,specificity,recall_malignant,recall_benign,tn,fp,fn,tp,errors,error_rate
4,SOB_M_PC_14-15687B,62,0.048387,1.0,0.048387,0.092308,NaN,NaN,0.048387,NaN,0,0,59,3,59,0.951613
16,SOB_B_F_14-29960AB,68,0.088235,0.0,0.000000,0.000000,NaN,0.088235,NaN,0.088235,6,62,0,0,62,0.911765
11,SOB_M_MC_14-10147,64,0.187500,1.0,0.187500,0.315789,NaN,NaN,0.187500,NaN,0,0,52,12,52,0.812500
21,SOB_B_A_14-22549G,130,0.307692,0.0,0.000000,0.000000,NaN,0.307692,NaN,0.307692,40,90,0,0,90,0.692308
6,SOB_M_MC_14-13418DE,55,0.309091,1.0,0.309091,0.472222,NaN,NaN,0.309091,NaN,0,0,38,17,38,0.690909
8,SOB_B_PT_14-21998AB,235,0.310638,0.0,0.000000,0.000000,NaN,0.310638,NaN,0.310638,73,162,0,0,162,0.689362
3,SOB_M_PC_14-9146,90,0.500000,1.0,0.500000,0.666667,NaN,NaN,0.500000,NaN,0,0,45,45,45,0.500000
28,SOB_B_PT_14-29315EF,60,0.683333,0.0,0.000000,0.000000,NaN,0.683333,NaN,0.683333,41,19,0,0,19,0.316667
22,SOB_B_A_14-29960CD,61,0.688525,0.0,0.000000,0.000000,NaN,0.688525,NaN,0.688525,42,19,0,0,19,0.311475
25,SOB_M_PC_15-190EF,65,0.692308,1.0,0.692308,0.818182,NaN,NaN,0.692308,NaN,0,0,20,45,20,0.307692


## 12. Integrazione della baseline CNN

La CNN baseline prodotta nel notebook 02 viene integrata soltanto se sono disponibili predizioni per immagine. Le sole metriche aggregate non permettono analisi coerenti per sottotipo o ingrandimento. Quando il file esiste, baseline ed EfficientNetB0 fine-tuned vengono confrontati sugli stessi sottogruppi e sullo stesso protocollo k-fold patient-wise; in caso contrario il notebook prosegue senza errore.

In [13]:
# Predizioni per immagine della CNN addestrata da zero nel notebook 02.
BASELINE_PREDICTIONS_PATH = RESULTS_DIR / "02_baseline_cnn" / "predictions" / "cnn_baseline_predictions.csv"

# Il confronto baseline è opzionale: se il file manca non viene sollevato un errore
# e tutte le analisi del modello transfer restano disponibili.
if not BASELINE_PREDICTIONS_PATH.is_file():
    print("Predizioni baseline non trovate: il confronto per sottogruppi con la baseline viene saltato.")
else:
    # Le metriche aggregate non bastano: servono probabilità associate alle singole immagini.
    baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_PATH)
    # Mantiene lo stesso protocollo k-fold patient-wise usato per il modello transfer;
    # mescolare protocolli diversi renderebbe il confronto metodologicamente scorretto.
    if "split_type" in baseline_predictions.columns:
        before = len(baseline_predictions)
        # Il filtro opera solo sul valore tecnico split_type già salvato nel CSV.
        baseline_predictions = baseline_predictions.loc[
            baseline_predictions["split_type"].astype(str).str.strip().str.lower().eq("kfold_patient_wise")
        ].copy()
        print(f"Predizioni baseline filtrate per kfold_patient_wise: {len(baseline_predictions)} / {before}")

    # Un file presente ma privo del protocollo richiesto non blocca il resto del notebook.
    if baseline_predictions.empty:
        print(
            "Attenzione: le predizioni baseline non contengono righe kfold_patient_wise; "
            "il confronto per sottogruppi con la baseline viene saltato."
        )
    else:
        # Metadati indispensabili per confrontare baseline e transfer negli stessi sottogruppi.
        required_baseline_metadata = ["subtype_name", "magnification"]
        # L'analisi per sottotipo e magnification richiede entrambe le colonne;
        # l'elenco identifica quali recuperare dal file metadata condiviso.
        missing_baseline_metadata = [
            column for column in required_baseline_metadata if column not in baseline_predictions.columns
        ]
        # Recupera soltanto i metadati assenti mediante relative_path.
        if missing_baseline_metadata:
            if "relative_path" not in baseline_predictions.columns:
                raise KeyError(
                    "Le predizioni baseline richiedono relative_path per recuperare i metadati dei sottogruppi."
                )
            metadata = pd.read_csv(METADATA_PATH)
            if "relative_path" not in metadata.columns:
                raise KeyError("Merge dei metadati non riuscito: relative_path è assente.")
            if metadata["relative_path"].duplicated().any():
                raise ValueError("I valori relative_path nei metadati devono essere univoci per il merge della baseline.")
            # validate="many_to_one" impone che ogni predizione trovi al massimo una riga metadata,
            # impedendo merge ambigui e moltiplicazioni artificiali delle osservazioni.
            baseline_predictions = baseline_predictions.merge(
                metadata[["relative_path", *missing_baseline_metadata]],
                on="relative_path", how="left", validate="many_to_one",
            )
        # binary_label è accettata come sorgente equivalente del target quando y_true manca.
        if "y_true" not in baseline_predictions.columns:
            if "binary_label" not in baseline_predictions.columns:
                raise KeyError("Le predizioni baseline richiedono y_true oppure binary_label.")
            baseline_predictions["y_true"] = baseline_predictions["binary_label"]
        # y_prob è necessaria per AUROC e per applicare in modo coerente la soglia 0.5.
        if "y_prob" not in baseline_predictions.columns:
            raise KeyError("Le predizioni baseline per immagine richiedono y_prob per le metriche dei sottogruppi.")
        baseline_predictions["y_true"] = pd.to_numeric(
            baseline_predictions["y_true"], errors="raise"
        ).astype(int)
        baseline_predictions["y_prob"] = pd.to_numeric(
            baseline_predictions["y_prob"], errors="raise"
        ).astype(float)
        # Controllo bloccante per evitare metriche probabilistiche su valori non validi.
        if baseline_predictions["y_prob"].isna().any() or not baseline_predictions["y_prob"].between(0, 1).all():
            raise ValueError("I valori y_prob della baseline devono essere finiti e compresi tra 0 e 1.")
        # Usa y_pred salvata; se assente, ricostruisce la stessa decisione a soglia 0.5.
        if "y_pred" in baseline_predictions.columns:
            baseline_predictions["y_pred"] = pd.to_numeric(
                baseline_predictions["y_pred"], errors="raise"
            ).astype(int)
        else:
            baseline_predictions["y_pred"] = (baseline_predictions["y_prob"] >= 0.5).astype(int)
        baseline_predictions["magnification"] = baseline_predictions["magnification"].map(
            normalize_magnification
        )

        # Costruisce un'unica tabella baseline vs EfficientNetB0 fine-tuned a partire
        # dalle predizioni per immagine, non da metriche aggregate non stratificabili.
        comparison_parts = []
        for source_name, source_df in [
            ("cnn_baseline_from_scratch", baseline_predictions),
            ("efficientnetb0_finetuned", best_predictions),
        ]:
            # Entrambi i modelli sono valutati sugli stessi sottogruppi, garantendo
            # un confronto coerente per sottotipo e ingrandimento.
            for subgroup in ("subtype_name", "magnification"):
                metrics = grouped_metrics(source_df, [subgroup])
                # subgroup_type distingue l'asse di analisi; subgroup_value conserva
                # il sottotipo o l'ingrandimento specifico in una colonna comune.
                metrics.insert(0, "subgroup_type", subgroup)
                metrics.insert(1, "subgroup_value", metrics.pop(subgroup).astype(str))
                # model_source rende la tabella filtrabile per baseline o modello transfer.
                metrics.insert(0, "model_source", source_name)
                comparison_parts.append(metrics)
        # La concatenazione conserva l'origine del modello per il confronto diretto.
        baseline_vs_best = pd.concat(comparison_parts, ignore_index=True)
        save_table(baseline_vs_best, "baseline_vs_best_transfer_subgroup_metrics.csv")
        display(baseline_vs_best)

Predizioni baseline filtrate per kfold_patient_wise: 2838 / 2838
Salvato: /Users/bertu/Projects/HistoBreastNet/results/04_d3_error_analysis/tables/baseline_vs_best_transfer_subgroup_metrics.csv


,model_source,subgroup_type,subgroup_value,n,accuracy,precision,recall,f1,auroc,specificity,recall_malignant,recall_benign,tn,fp,fn,tp
0,cnn_baseline_from_scratch,subtype_name,adenosis,444,0.993243,0.000000,0.000000,0.000000,NaN,0.993243,NaN,0.993243,441,3,0,0
1,cnn_baseline_from_scratch,subtype_name,tubular_adenoma,247,0.712551,0.000000,0.000000,0.000000,NaN,0.712551,NaN,0.712551,176,71,0,0
2,cnn_baseline_from_scratch,subtype_name,papillary_carcinoma,291,0.422680,1.000000,0.422680,0.594203,NaN,NaN,0.422680,NaN,0,0,168,123
3,cnn_baseline_from_scratch,subtype_name,ductal_carcinoma,219,0.840183,1.000000,0.840183,0.913151,NaN,NaN,0.840183,NaN,0,0,35,184
4,cnn_baseline_from_scratch,subtype_name,mucinous_carcinoma,271,0.678967,1.000000,0.678967,0.808791,NaN,NaN,0.678967,NaN,0,0,87,184
5,cnn_baseline_from_scratch,subtype_name,phyllodes_tumor,453,0.392936,0.000000,0.000000,0.000000,NaN,0.392936,NaN,0.392936,178,275,0,0
6,cnn_baseline_from_scratch,subtype_name,fibroadenoma,287,0.494774,0.000000,0.000000,0.000000,NaN,0.494774,NaN,0.494774,142,145,0,0
7,cnn_baseline_from_scratch,subtype_name,lobular_carcinoma,626,0.857827,1.000000,0.857827,0.923474,NaN,NaN,0.857827,NaN,0,0,89,537
8,cnn_baseline_from_scratch,magnification,100X,743,0.694482,0.666667,0.782842,0.720099,0.745337,0.605405,0.782842,0.605405,224,146,81,292
9,cnn_baseline_from_scratch,magnification,40X,712,0.661517,0.651226,0.678977,0.664812,0.698422,0.644444,0.678977,0.644444,232,128,113,239


## 13. Sintesi interpretativa per D3

- Il modello transfer selezionato viene valutato rispetto a sottotipo tumorale e fattore di ingrandimento.
- La robustezza per sottotipo è sintetizzata principalmente tramite la recall della classe rilevante; la F1 positiva resta una vista secondaria perché non è informativa nei sottotipi esclusivamente benigni.
- L'AUROC non è definita nei gruppi mono-classe e le stime dei sottogruppi con supporto ridotto richiedono cautela.
- L'analisi per ingrandimento verifica la stabilità tra 40X, 100X, 200X e 400X.
- I falsi negativi vengono isolati perché rappresentano casi maligni non riconosciuti.
- L'analisi della confidenza distingue gli errori incerti da quelli commessi con elevata sicurezza.
- L'analisi patient-level verifica se gli errori si concentrano in pochi pazienti non visti.

Le conclusioni quantitative devono essere lette dalle tabelle e dalle figure salvate dal notebook. La sintesi deve considerare congiuntamente supporto, metriche per sottogruppo, distribuzione degli errori, confidenza e concentrazione patient-level.